# 🌱 GA Potager — atelier data scientist

Workflow du notebook :
1. **Configurer le bac** dans le bloc de code ci-dessous (dimensions, terre, substrats…)
2. **Lancer l'algorithme génétique** (la lib Rust via le CLI en mode `--json`)
3. **Lire le gagnant** et *pourquoi* il gagne (décomposition du score)
4. **Explorer toute la population** : les bonnes propositions, et les moins bonnes

## 1. Configuration du bac

Modifie librement les valeurs, puis réexécute cette cellule (et les suivantes).

In [ ]:
import json
from pathlib import Path

# --- Ton bac, à toi de jouer ---
BAC = {
    "id": "reference",
    "nom": "Bac de référence",
    "dimensions_cm": {"longueur": 120, "largeur": 80, "hauteur": 50},
    "hauteur_terre_cm": 40,
    "marge_bord_cm": 5,
    "protection": "exterieur",
    "substrats": ["terreau", "compost", "fibre_de_coco"],
    "compost_disponible": True,
    "drainage_eau": True,
    "exposition": "plein_soleil",
    "saison_cible": "ete",
    "temp_moyenne_c": {"min": 15, "max": 30},
}

# Où vit le repo (le notebook est dans notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

# Écrit bacs.json : c'est L'entrée de l'AG
(DATA / "bacs.local.json").write_text(
    json.dumps([BAC], indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"bacs.local.json écrit (hors git) : {BAC['nom']}")

## 2. Lancement de l'algorithme génétique

Le binaire Rust est compilé si besoin, puis lancé en mode `--json`. Change `SEED` pour explorer la variabilité des runs.

In [ ]:
import subprocess, sys

SEED = 42  # change pour explorer la variabilité

BIN = ROOT / "ga-plantes" / "target" / "debug" / ("ga-potager-cli.exe" if sys.platform == "win32" else "ga-potager-cli")
if not BIN.exists():
    print("Compilation du binaire...")
    subprocess.run(["cargo", "build", "--manifest-path", str(ROOT / "ga-plantes" / "Cargo.toml")], check=True)

proc = subprocess.run(
    [str(BIN), "--data-dir", str(DATA), "--config-path", str(DATA / "config.json"), "--seed", str(SEED), "--json", "--bacs", str(DATA / "bacs.local.json")],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
)
if proc.returncode != 0:
    print(proc.stderr)  # erreurs de données : la validation explique quoi corriger
    raise SystemExit(1)

resultat = json.loads(proc.stdout[proc.stdout.index("{"):])
print(f"Évolution terminée en {resultat['generations']} générations (seed {resultat['seed']})")

## 3. Le gagnant — et *pourquoi* il gagne

Le score total est la somme de trois composantes pondérées. Le graphe montre laquelle porte la solution.

In [ ]:
import matplotlib.pyplot as plt

meilleur = resultat["meilleur"]
print("🏆 Potager gagnant — bac", meilleur["bac"])
for p in meilleur["plantes"]:
    print(f"   - {p['nom']} ({p['famille']})")
print(f"Score total : {meilleur['score']:.2f}")
print("\nInteractions appliquées :")
for i in meilleur["detail"]["interactions_actives"] or ["(aucune)"]:
    print(f"   {i}")

d = meilleur["detail"]
composantes = {"Rendement (x5)": d["rendement"], "Interactions (x7)": d["interactions"], "Occupation (x10)": d["occupation"]}
couleurs = ["#4caf50" if v >= 0 else "#e53935" for v in composantes.values()]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(composantes.keys()), list(composantes.values()), color=couleurs)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Pourquoi ce potager gagne — décomposition du score (total {meilleur['score']:.2f})")
for i, v in enumerate(composantes.values()):
    ax.text(v, i, f" {v:+.2f}", va="center")
plt.tight_layout(); plt.show()

## 4. Toute la population — les bonnes et les moins bonnes propositions

L'AG ne produit pas qu'un gagnant : la dernière génération contient toutes les solutions évaluées. L'histogramme montre la distribution, le classement les extrêmes.

In [ ]:
pop = sorted(resultat["population"], key=lambda x: -x["score"])
scores = [x["score"] for x in pop]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(scores, bins=20, color="#66bb6a", edgecolor="white")
ax.axvline(meilleur["score"], color="#1b5e20", linestyle="--", label=f"gagnant ({meilleur['score']:.1f})")
ax.set_xlabel("Score"); ax.set_ylabel("Nombre d'individus")
ax.set_title(f"Distribution des {len(pop)} individus de la population finale")
ax.legend(); plt.tight_layout(); plt.show()

print("🥇 Top 3 :")
for x in pop[:3]:
    print(f"   {x['score']:8.2f}  {' + '.join(x['plantes'])}")
print("\n💀 Flop 3 :")
for x in pop[-3:]:
    print(f"   {x['score']:8.2f}  {' + '.join(x['plantes']) or '(bac vide)'}")

## 🗺️ Le gagnant, vu du dessus

Schéma du bac (échelle réelle) : chaque plante est un cercle de rayon = la moitié de son espacement minimum. Si des cercles se chevauchent ou sortent du cadre, la combinaison est à la limite de la contrainte de surface — exactement ce que la validation et la fitness arbitrent.

In [ ]:
import json as _json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

# Espacements réels depuis plantes.json
_plantes_data = _json.loads((DATA / "plantes.json").read_text(encoding="utf-8"))
espacements = {p["id"]: p["espacement_min_cm"] for p in _plantes_data}
familles = {p["id"]: p["famille"] for p in _plantes_data}

L, l = BAC["dimensions_cm"]["longueur"], BAC["dimensions_cm"]["largeur"]
marge = BAC["marge_bord_cm"]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.add_patch(Rectangle((0, 0), L, l, fill=False, edgecolor="black", linewidth=2))
ax.add_patch(Rectangle((marge, marge), L - 2*marge, l - 2*marge, fill=False, edgecolor="#2e7d32", linewidth=1, linestyle="--"))

COULEURS = {"Solanacées": "#e53935", "Fabacées": "#43a047", "Brassicacées": "#8e24aa", "Amaryllidacées": "#795548", "Lamiacées": "#00897b", "Apiacées": "#fb8c00", "Astéracées": "#fdd835", "Cucurbitacées": "#5e35b1"}

# Placement en rangées DANS la surface utile : chaque plante occupe un
# carré de côté = son espacement ; l'avance verticale entre rangées est
# la hauteur de la rangée précédente (sans double-comptage).
x, y = marge, marge
h_rangee = 0
deborde = []
for p in meilleur["plantes"]:
    esp = espacements[p["id"]]
    r = esp / 2
    if x + esp > L - marge:            # nouvelle rangée
        x = marge
        y += h_rangee                   # collée sous la précédente
        h_rangee = 0
    if y + esp > l - marge:            # déborderait du bas : on le signale
        deborde.append(p["id"])
        continue
    ax.add_patch(Circle((x + r, y + r), r, color=COULEURS.get(familles[p["id"]], "gray"), alpha=0.7))
    # label sous le cercle pour les petites plantes (illisibles au centre)
    if esp >= 30:
        ax.text(x + r, y + r, p["id"], ha="center", va="center", fontsize=9)
    else:
        ax.text(x + r, y - 1.5, p["id"], ha="center", va="top", fontsize=7)
    x += esp
    h_rangee = max(h_rangee, esp)

if deborde:
    print(f"⚠️ Placement illustratif incomplet (rangées trop serrées pour tout caser) : {', '.join(deborde)}")
    print("La combinaison reste VALIDE : la contrainte du Rust est la somme des carrés d'espacement ≤ surface utile. Ce dessin est indicatif, pas un plan de plantation.")

ax.set_xlim(-5, L + 5); ax.set_ylim(-5, l + 5)
ax.set_aspect("equal")
ax.set_title(f"Potager gagnant — bac {L}x{l} cm (schéma indicatif, échelle réelle)")
plt.tight_layout(); plt.show()

## 📈 Évolution de la population au fil des générations

Deux lectures du même historique :
- **Convergence** : fitness min/moy/max par génération. Si le max monte encore à la fin → early stopping trop pressé ; s'il stagne dès la génération 2-3 → population trop petite ou espace trop simple
- **Composition** : % d'individus contenant chaque plante, par génération. Une plante qui part haute et monte = imposée par la sélection ; une plante qui émerge tard = trouvée par la mutation ; une plante qui disparaît = éliminée

In [ ]:
import matplotlib.pyplot as plt

hist = resultat["historique"]
taille_pop = len(resultat["population"])
gens = [h["generation"] for h in hist]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

# 1. Convergence de la fitness
ax1.plot(gens, [h["fitness_min"] for h in hist], alpha=0.5, label="min", color="#999999")
ax1.plot(gens, [h["fitness_moyenne"] for h in hist], label="moyenne", color="#1976d2", linewidth=2)
ax1.plot(gens, [h["fitness_max"] for h in hist], label="max", color="#2e7d32", linewidth=2)
ax1.set_ylabel("Fitness")
ax1.set_title(f"Convergence sur {len(hist)} générations (population {taille_pop})")
ax1.legend()

# 2. Composition : % d'individus contenant chaque plante
plantes_ids = sorted({pid for h in hist for pid, _ in h["presence_plantes"]})
for pid in plantes_ids:
    serie = [100 * dict(h["presence_plantes"]).get(pid, 0) / taille_pop for h in hist]
    epaisseur = 2.5 if pid == "tomate" else 1.2
    ax2.plot(gens, serie, label=pid, linewidth=epaisseur)
ax2.set_xlabel("Génération"); ax2.set_ylabel("% de la population")
ax2.set_title("Présence de chaque plante dans la population")
ax2.legend(ncol=3, fontsize=8)

plt.tight_layout(); plt.show()

## 5. Comment itérer

- **Change une chose à la fois** : une valeur du bac, la seed, ou une plante dans `plantes.json`
- **Compare les runs** : si le gagnant change du tout au tout entre deux seeds, l'AG explore mal (population trop petite, early stopping trop pressé)
- **Lis le flop** : les mauvaises propositions sont aussi instructives — une compétition pénalisée, un bac quasi vide, un rendement faible
- Les **poids** (5 / 7 / 10) vivent dans `fitness/eval.rs` : changer le `7` en `12` valorise le compagnonnage, puis relance et compare les histogrammes